In [1]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel
import torch.nn.functional as F

C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class RobertaMultiClassScorer(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        output = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = output.pooler_output
        x = self.dropout(pooled)
        return self.classifier(x)  # Raw logits

In [3]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_classes = 4  # adjust if different
model = RobertaMultiClassScorer(num_classes=num_classes).to(device)

state_dict = torch.load('roberta_multiclass_model.pth', map_location=device)
model.load_state_dict(state_dict, strict=False)

Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.decoder.weight', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


_IncompatibleKeys(missing_keys=['roberta.embeddings.position_ids'], unexpected_keys=[])

In [5]:
class_to_index = {'Anxiety': 0, 'Depression': 1, 'Normal': 2, 'Stress': 3}
index_to_class = {v: k for k, v in class_to_index.items()}

In [6]:
def predict(text, model, tokenizer, device, index_to_class):
    model.eval()
    with torch.no_grad():
        inputs = tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=128
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        logits = model(inputs['input_ids'], inputs['attention_mask'])
        probs = torch.softmax(logits, dim=1).squeeze().cpu()

        predicted_index = torch.argmax(probs).item()
        predicted_class = index_to_class[predicted_index]

        result = {
            index_to_class[i]: round(prob.item() * 100, 2)
            for i, prob in enumerate(probs)
        }
        return predicted_class, result

In [7]:
text_input = "I'm sometimes worried something will go wrong, even when everything seems okay"

predicted_class, scores = predict(text_input, model, tokenizer, device, index_to_class)

print(f"\nText: {text_input}")
print(f"Predicted Class: {predicted_class}")
print("\nClass Probabilities (%):")
for cls, score in scores.items():
    print(f"{cls}: {score}%")

print(scores)


Text: I'm sometimes worried something will go wrong, even when everything seems okay
Predicted Class: Anxiety

Class Probabilities (%):
Anxiety: 76.28%
Depression: 4.16%
Normal: 15.65%
Stress: 3.91%
{'Anxiety': 76.28, 'Depression': 4.16, 'Normal': 15.65, 'Stress': 3.91}


In [8]:
from gym import Env
from gym.spaces import Discrete, MultiDiscrete
import numpy as np
import random

In [9]:
anxiety_questions = ['How often have you felt afraid, as if something awful might happen?', 'How often have you been so restless due to academic pressure that it is hard to sit still?','How often have you worried too much about academic affairs?','How often have you been easily annoyed or irritated because of academic pressure?','How often have you had trouble relaxing due to academic pressure?','How often have you been unable to stop worrying about your academic affairs?','How often you felt nervous, anxious or on edge due to academic pressure? ']
depression_questions = ['How often have you had thoughts that you would be better off dead, or of hurting yourself?','How often have you moved or spoke too slowly for other people to notice? Or you have been moving a lot more than usual because you have been restless?','How often have you been having trouble concentrating on things, such as reading the books or watching television?','how often have you been feeling bad about yourself - or that you are a failure or have let yourself or your family down?','How often have you had poor appetite or overeating?','How often have you been feeling tired or having little energy?','How often have you had trouble falling or staying asleep, or sleeping too much?','How often have you been feeling down, depressed or hopeless?','How often have you had little interest or pleasure in doing things?']
stress_questions = ['How often you felt as if academic difficulties are piling up so high that you could not overcome them?','How often you got angered due to bad performance or low grades that is beyond your control?','How often you felt as if your academic performance was on top?','How often are you able to control irritations in your academic / university affairs?','How often you felt as if things in your academic life is going on your way?','How often you felt confident about your ability to handle your academic / university problems?','How often you felt as if you could not cope with all the mandatory academic activities? (e.g, assignments, quiz, exams)','How often you felt nervous and stressed because of academic pressure?','How often you felt as if you were unable to control important things in your academic affairs?','How often have you felt upset due to something that happened in your academic affairs?']

In [10]:
questions_arr = [anxiety_questions, depression_questions, stress_questions]

In [11]:
anxiety_threshold = [10,9,8,6,5,3,2]
depression_threshold = [10,10,9,8,7,6,4,3,2]
stress_threshold = [23,20,17,16,15,14,12,9,6,3]

In [12]:
def askQuestion(ind, disorderOffset):
    print(questions_arr[disorderOffset][ind-1])

    text_input = input("Your Response : ")
    
    _, probs = predict(text_input, model, tokenizer, device, index_to_class)
    
    anxiety_score = int(round(probs.get('Anxiety', 0) / 100 * 3))
    depression_score = int(round(probs.get('Depression', 0) / 100 * 3))
    stress_score = int(round(probs.get('Stress', 0) / 100 * 4))
    normal_score = int(round(probs.get('Normal', 0) / 100 * 3))
    
    return [anxiety_score, depression_score, stress_score, normal_score]

In [13]:
class ChatBotEnv(Env):
  def __init__(self):
    self.action_space = Discrete(4)
    self.observation_space = MultiDiscrete([4,4,5,4])
    self.state = [random.randint(0,3),random.randint(0,3),random.randint(0,4),random.randint(0,3)]
    self.anxiety_question_offset = 7
    self.depression_question_offset = 9
    self.stress_question_offset = 10
    self.anxiety_score = 0
    self.depression_score = 0
    self.stress_score = 0
    self.anxiety_predicted = False
    self.depression_predicted = False
    self.stress_predicted = False

  def step(self, action):
    done = False
      
    # Ask Anxiety
    if action == 0:
        if self.anxiety_question_offset == -1:
            reward = -5
        elif self.anxiety_question_offset == 0 and self.anxiety_score >= anxiety_threshold[0]:
            self.anxiety_predicted = True
            self.anxiety_question_offset = -1
            reward = 10
        elif self.anxiety_question_offset == 7 or self.anxiety_score >= anxiety_threshold[self.anxiety_question_offset]:
            self.state = askQuestion(self.anxiety_question_offset, 0)
            self.anxiety_question_offset -= 1;
            self.anxiety_score += self.state[0]
            reward = -1
        else:
            reward = -5
    # Ask Depression
    if action == 1:
        if self.depression_question_offset == -1:
            reward = -5
        elif self.depression_question_offset == 0 and self.depression_score >= depression_threshold[0]:
            self.depression_predicted = True
            self.depression_question_offset = -1
            reward = 10
        elif self.depression_question_offset == 9 or self.depression_score >= depression_threshold[self.depression_question_offset]:
            self.state = askQuestion(self.depression_question_offset, 1)
            self.depression_question_offset -= 1;
            self.depression_score += self.state[1]
            reward = -1
        else:
            reward = -5
    # Ask Stress
    if action == 2:
        if self.stress_question_offset == -1:
            reward = -5
        elif self.stress_question_offset == 0 and self.stress_score >= stress_threshold[0]:
            self.stress_predicted = True
            self.stress_question_offset = -1
            reward = 10
        elif self.stress_question_offset == 10 or self.stress_score >= stress_threshold[self.stress_question_offset]:
            self.state = askQuestion(self.stress_question_offset, 2)
            self.stress_question_offset -= 1;
            self.stress_score += self.state[2]
            reward = -1
        else:
            reward = -5
    else:
        good_to_go = True
        if not self.anxiety_predicted:
            if self.anxiety_question_offset == 7:
                reward = -5
                good_to_go = False
            elif self.anxiety_score >= anxiety_threshold[self.anxiety_question_offset]:
                reward = -5
                good_to_go = False
        if not self.depression_predicted:
            if self.depression_question_offset == 9:
                reward = -5
                good_to_go = False
            elif self.depression_score >= depression_threshold[self.depression_question_offset]:
                reward = -5
                good_to_go = False
        if not self.stress_predicted:
            if self.stress_question_offset == 10:
                reward = -5
                good_to_go = False
            elif self.stress_score >= stress_threshold[self.stress_question_offset]:
                reward = -5
                good_to_go = False
        if good_to_go:
            done = True
            if self.anxiety_predicted:
                print("Anxiety")
            if self.depression_predicted:
                print("Depression")
            if self.stress_predicted:
                print("Stress")
            if not self.anxiety_predicted and not self.depression_predicted and not self.stress_predicted:
                print("Normal")
            reward = 20
            
    info = {}
        
    return self.state, reward, done, info

  def render(self):
    pass

  def reset(self):
    disorder_scores = [0,0,0]
    self.state = [random.randint(0,3),random.randint(0,3),random.randint(0,4),random.randint(0,3)]
    self.anxiety_question_offset = 7
    self.depression_question_offset = 9
    self.stress_question_offset = 10
    self.anxiety_score = 0
    self.depression_score = 0
    self.stress_score = 0
    self.anxiety_predicted = False
    self.depression_predicted = False
    self.stress_predicted = False
      
    return self.state

In [14]:
env = ChatBotEnv()

In [16]:
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines.common.policies import MlpLstmPolicy

In [17]:
import os

In [18]:
path = os.path.join("Training_Final","Recurrent_Model_Final")

In [19]:
rl_model = RecurrentPPO.load(path,env)

C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [20]:
device = "cpu"

In [21]:
import gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO  

obs = env.reset()
done = False

while not done:
    action, _ = rl_model.predict(obs)
    obs, reward, done, info = env.step(action)

How often have you had little interest or pleasure in doing things?


Your Response :  Um… pretty often, I guess… even when I try to do stuff I usually like, it just… feels flat, and I can’t really get into it…


How often you felt nervous, anxious or on edge due to academic pressure? 


Your Response :  Oh… all the time, honestly… I keep thinking I’m behind, or that I’ll mess something up… it’s like my brain never stops racing.


How often have you been feeling down, depressed or hopeless?


...

How often have you moved or spoke too slowly for other people to notice? Or you have been moving a lot more than usual because you have been restless?


Your Response :  Oh… like, almost every day… sometimes I feel so restless I can’t sit still, and other times I just… drag along and feel like I’m moving in slow motion… it’s exhausting.


How often have you had thoughts that you would be better off dead, or of hurting yourself?


Your Response :  Never had thoughts this negative, but yeah negative enough to keep me up at night


Depression
